# M2 — Elektra · 4 cenários do estudo de caso (Google OR-Tools)
**Treinamento de Otimização · Genoa para Gradus · M2 — Otimização de Rede**

Modelo de transporte com transbordo: 2 plantas → 3 CDs → 9 regiões.
Os 4 cenários do estudo de caso, parametrizados — basta uma função e um loop.


In [ ]:
!pip install ortools -q

## 1) Dados do caso

In [ ]:
plantas = ['Sorocaba', 'Contagem']
cds     = ['Ribeirao', 'Cajamar', 'Uberlandia']
regioes = ['A','B','C','D','E','F','G','H','I']

oferta  = {'Sorocaba': 30000, 'Contagem': 20000}
demanda = {'A':6300,'B':4880,'C':2130,'D':1210,'E':6120,'F':4830,'G':2750,'H':8580,'I':4460}
c_prod  = {'Sorocaba': 21.0, 'Contagem': 20.0}

# Custos planta -> CD (rotas vetadas: ausentes do dict)
c_pc = {
  ('Sorocaba','Ribeirao'):6.40, ('Sorocaba','Cajamar'):4.40, ('Sorocaba','Uberlandia'):8.30,
  ('Contagem','Cajamar'):7.80,  ('Contagem','Uberlandia'):2.40,
}

# Custos CD -> região (rotas vetadas: ausentes do dict)
c_cr = {
  ('Ribeirao','A'):0.60,('Ribeirao','B'):4.20,('Ribeirao','C'):6.20,('Ribeirao','D'):8.80,('Ribeirao','E'):12.00,
  ('Cajamar','A'):10.40,('Cajamar','B'):10.80,('Cajamar','C'):9.00,('Cajamar','D'):12.00,
  ('Cajamar','E'):5.40,('Cajamar','F'):5.40,('Cajamar','G'):6.80,('Cajamar','H'):6.60,('Cajamar','I'):3.40,
  ('Uberlandia','E'):10.80,('Uberlandia','F'):6.60,('Uberlandia','G'):4.80,('Uberlandia','H'):4.20,('Uberlandia','I'):5.00,
}

# Designação atual (Cenário 1): cada CD atende um conjunto fixo
designacao_atual = {
  'Ribeirao':   ['A','B','C','D'],
  'Cajamar':    ['E','F','G'],
  'Uberlandia': ['H','I'],
}

# Frete direto (Cenário 3): planta -> região, novas rotas
c_pr_diretas = {
  ('Contagem','H'):0.60,
  ('Contagem','I'):1.40,
  ('Sorocaba','B'):7.00,
}

## 2) Função genérica que roda o modelo
Recebe parâmetros (designação fixa, fretes diretos, fator de crescimento, capacidade extra) e devolve custo + solução.

In [ ]:
from ortools.linear_solver import pywraplp

def resolver(designacao=None, c_pr=None, fator_demanda=1.0, capacidade_extra=None):
    """
    designacao: dict CD->[regiões] forçando designação fixa (Cenário 1). None = livre.
    c_pr:       dict (planta,região)->custo de frete direto. None = sem frete direto.
    fator_demanda: multiplica a demanda das regiões E..I (Cenário 4).
    capacidade_extra: dict planta->capacidade adicional (Cenário 4 expansão).
    """
    # Demanda ajustada
    dem = dict(demanda)
    if fator_demanda != 1.0:
        for k in ['E','F','G','H','I']:
            dem[k] = int(dem[k] + 6000 * (fator_demanda - 1.0) / 1.0) if fator_demanda > 1 else dem[k]
    # Para o Cenário 4 com +6000 cada região E-I, podemos receber fator_demanda como flag
    
    # Capacidade ajustada
    cap = dict(oferta)
    if capacidade_extra:
        for i,extra in capacidade_extra.items():
            cap[i] += extra

    solver = pywraplp.Solver.CreateSolver('GLOP')
    x = {(i,j): solver.NumVar(0, solver.infinity(), f'x_{i}_{j}') for (i,j) in c_pc}
    y = {(j,k): solver.NumVar(0, solver.infinity(), f'y_{j}_{k}') for (j,k) in c_cr}
    z = {}
    if c_pr:
        z = {(i,k): solver.NumVar(0, solver.infinity(), f'z_{i}_{k}') for (i,k) in c_pr}

    # Capacidade da planta (inclui frete direto se houver)
    for i in plantas:
        total_planta = sum(x[(i,j)] for j in cds if (i,j) in x)
        if z:
            total_planta += sum(z[(i,k)] for k in regioes if (i,k) in z)
        solver.Add(total_planta <= cap[i])

    # Conservação no CD
    for j in cds:
        entra = sum(x[(i,j)] for i in plantas if (i,j) in x)
        sai   = sum(y[(j,k)] for k in regioes if (j,k) in y)
        solver.Add(entra == sai)

    # Demanda da região
    for k in regioes:
        recebido = sum(y[(j,k)] for j in cds if (j,k) in y)
        if z:
            recebido += sum(z[(i,k)] for i in plantas if (i,k) in z)
        solver.Add(recebido == dem[k])

    # Designação fixa CD<->região (Cenário 1)
    if designacao:
        for j in cds:
            atendidas = designacao.get(j, [])
            for k in regioes:
                if k not in atendidas and (j,k) in y:
                    solver.Add(y[(j,k)] == 0)

    # FO: produção + transporte planta-CD + transporte CD-região + frete direto
    prod = sum(c_prod[i] * (
                  sum(x[(i,j)] for j in cds if (i,j) in x) +
                  (sum(z[(i,k)] for k in regioes if (i,k) in z) if z else 0)
              ) for i in plantas)
    t_pc = sum(c_pc[(i,j)] * x[(i,j)] for (i,j) in x)
    t_cr = sum(c_cr[(j,k)] * y[(j,k)] for (j,k) in y)
    t_dir = sum(c_pr[(i,k)] * z[(i,k)] for (i,k) in z) if z else 0
    solver.Minimize(prod + t_pc + t_cr + t_dir)

    status = solver.Solve()
    if status != pywraplp.Solver.OPTIMAL:
        return None
    return {
        'custo': solver.Objective().Value(),
        'x': {k: v.solution_value() for k,v in x.items() if v.solution_value() > 0.01},
        'y': {k: v.solution_value() for k,v in y.items() if v.solution_value() > 0.01},
        'z': {k: v.solution_value() for k,v in z.items() if v.solution_value() > 0.01} if z else {},
    }

## 3) Rodando os 4 cenários

In [ ]:
def fmt(v):
    return f'R$ {v:,.2f}'.replace(',','X').replace('.',',').replace('X','.')

# Cenário 1: situação atual
r1 = resolver(designacao=designacao_atual)
print(f'CENÁRIO 1 (atual)         : {fmt(r1["custo"])}')

# Cenário 2: livre designação
r2 = resolver()
print(f'CENÁRIO 2 (livre)         : {fmt(r2["custo"])}   economia: {fmt(r1["custo"]-r2["custo"])} ({100*(r1["custo"]-r2["custo"])/r1["custo"]:.1f}%)')

# Cenário 3: livre + frete direto
r3 = resolver(c_pr=c_pr_diretas)
print(f'CENÁRIO 3 (+ frete direto): {fmt(r3["custo"])}   economia adic.: {fmt(r2["custo"]-r3["custo"])} ({100*(r2["custo"]-r3["custo"])/r2["custo"]:.1f}%)')

# Cenário 4: crescimento de demanda (E-I +6000 cada)
demanda_4 = dict(demanda)
for k in ['E','F','G','H','I']:
    demanda_4[k] += 6000
# Para Cenário 4, modificamos demanda diretamente. Como nossa função usa o dict global, vamos hackear:
demanda_orig = dict(demanda)
demanda.update(demanda_4)
r4_sem_expansao = resolver()
print(f'CENÁRIO 4 (cresc. sem exp): {"INVIÁVEL" if r4_sem_expansao is None else fmt(r4_sem_expansao["custo"])}  ← capacidade insuficiente!')

# Cenário 4 com expansão de Contagem (+25k)
r4_com_expansao = resolver(capacidade_extra={'Contagem': 25000})
print(f'CENÁRIO 4 (+ exp Contagem): {fmt(r4_com_expansao["custo"])}')

# Restaurar demanda original
demanda.clear(); demanda.update(demanda_orig)

## 4) Sensibilidade — preços-sombra das restrições de capacidade
Os preços-sombra das restrições de capacidade dizem **quanto vale 1 unidade adicional** de capacidade em cada planta. É exatamente o que o diretor precisa para defender um investimento em expansão.

In [ ]:
# (Implementação: re-rodar o modelo capturando as restrições e seus dual_values.
#  Veja https://developers.google.com/optimization/lp/glop para sensitivity report no OR-Tools.)
print('Veja a planilha caso 1 elektra prob transportes sol.xlsx — Relatório de Sensibilidade do Solver.')

## Conclusão
- **Cenário 2** (abrir designação CD↔região) é a maior alavanca de economia — sem investimento.
- **Cenário 3** (frete direto Contagem→H/I) traz economia adicional pela proximidade geográfica.
- **Cenário 4** (crescimento) torna **expansão de Contagem** essencial — capacidade total atual de 50k é insuficiente para a demanda futura de ~71k.

Todos os 4 cenários rodam em milissegundos. Esse é o ganho de Python aqui: parametrizar e iterar é trivial. No Excel, são 4 cliques manuais e 4 cópias de planilha.